<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-07-anomaly-triage-and-embeddings-for-lumina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7 (graded) — Anomaly triage + reusable embeddings for Lumina
**Course 1: Hands-On Deep Learning with Python — Chapter 7: Autoencoders & embeddings**

**Problem brief (Dr. Ana Reyes & a Lumina Health research lead):** "We can't label every
unusual patient encounter, but we want the ones worth a human review surfaced. And we want a
dense patient representation the downstream readmission model can reuse."
Target: recall@5% on known-anomalous encounters; embeddings that lift the downstream AUC.

**What you'll submit:** an autoencoder ranking encounters by reconstruction error, a
recall@5% result, and a downstream classifier comparison (raw features vs. AE embeddings).

## 1. Load the data (with offline fallback)

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(0)

def load_diabetes_data():
    try:
        import kagglehub
        path = kagglehub.dataset_download('brandao/diabetes')
        df = pd.read_csv(f'{path}/diabetic_data.csv')
        print('Loaded the real Diabetes 130-US Hospitals dataset:', df.shape)
        return df
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — a synthetic encounter table with a similar shape.')
        n = 5000
        return pd.DataFrame({
            'time_in_hospital': np.random.randint(1, 14, n),
            'num_lab_procedures': np.random.randint(1, 100, n),
            'num_procedures': np.random.randint(0, 6, n),
            'num_medications': np.random.randint(1, 40, n),
            'number_outpatient': np.random.poisson(0.3, n),
            'number_emergency': np.random.poisson(0.2, n),
            'number_inpatient': np.random.poisson(0.4, n),
            'number_diagnoses': np.random.randint(1, 16, n),
            'readmitted': np.random.choice(['NO', '>30', '<30'], n, p=[0.55, 0.35, 0.10]),
        })

df = load_diabetes_data()
num_cols = [c for c in ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                          'num_medications', 'number_outpatient', 'number_emergency',
                          'number_inpatient', 'number_diagnoses'] if c in df.columns]
df = df.dropna(subset=num_cols)
X = df[num_cols].to_numpy(dtype=np.float32)
X = (X - X.mean(0)) / (X.std(0) + 1e-8)

# a stand-in "known anomalous" label for evaluation: encounters with an emergency+inpatient
# history in the extreme tail — this is what recall@5% is measured against below
anomaly_score_proxy = df['number_emergency'].to_numpy() + df['number_inpatient'].to_numpy()
known_anomalous = anomaly_score_proxy >= np.percentile(anomaly_score_proxy, 95)
y_readmit = (df['readmitted'] != 'NO').astype(int).to_numpy() if 'readmitted' in df.columns else np.zeros(len(df), dtype=int)

print('X:', X.shape, 'known-anomalous rate:', known_anomalous.mean().round(3))

## 2. Autoencoder (plain + denoising variant)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

class Autoencoder(nn.Module):
    def __init__(self, n_features, bottleneck=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 16), nn.ReLU(), nn.Linear(16, n_features),
        )

    def forward(self, x, noise_std=0.0):
        # noise_std > 0 during training = a denoising autoencoder
        x_in = x + torch.randn_like(x) * noise_std if noise_std > 0 else x
        z = self.encoder(x_in)
        return self.decoder(z), z


X_t = torch.tensor(X, dtype=torch.float32)
loader = DataLoader(TensorDataset(X_t), batch_size=128, shuffle=True)

ae = Autoencoder(X.shape[1]).to(device)
optimizer = torch.optim.AdamW(ae.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(30):
    ae.train()
    epoch_loss, nb = 0.0, 0
    for (xb,) in loader:
        xb = xb.to(device)
        optimizer.zero_grad()
        recon, _ = ae(xb, noise_std=0.1)  # denoising: reconstruct the CLEAN input from a noisy one
        loss = loss_fn(recon, xb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item(); nb += 1
    if epoch % 5 == 0 or epoch == 29:
        print(f'epoch {epoch}: recon MSE = {epoch_loss / nb:.4f}')

## 3. Rank by reconstruction error — recall@5%

In [ ]:
ae.eval()
with torch.no_grad():
    recon, embeddings = ae(X_t.to(device), noise_std=0.0)
    recon_error = ((recon.cpu() - X_t) ** 2).mean(dim=1).numpy()

k = max(1, int(len(recon_error) * 0.05))
top_k_idx = np.argsort(-recon_error)[:k]
caught = known_anomalous[top_k_idx].sum()
recall_at_5pct = caught / known_anomalous.sum()
print(f'Recall@5% (reconstruction-error ranking): {recall_at_5pct:.3f}')
print(f'({caught} of {known_anomalous.sum()} known-anomalous encounters caught in the top {k})')

## 4. Do the embeddings help a downstream model?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

emb = embeddings.cpu().numpy()
X_raw_tr, X_raw_te, X_emb_tr, X_emb_te, y_tr, y_te = train_test_split(
    X, emb, y_readmit, test_size=0.25, random_state=0, stratify=y_readmit if y_readmit.sum() > 0 else None
)

def fit_and_auc(X_tr, X_te, y_tr, y_te, label):
    if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
        print(f'{label}: not enough label variation to fit (offline fallback data) — skipped.')
        return None
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
    print(f'{label}: AUC = {auc:.4f}')
    return auc

auc_raw = fit_and_auc(X_raw_tr, X_raw_te, y_tr, y_te, 'Raw features only')
auc_raw_plus_emb = fit_and_auc(
    np.hstack([X_raw_tr, X_emb_tr]), np.hstack([X_raw_te, X_emb_te]), y_tr, y_te,
    'Raw features + AE embeddings',
)
if auc_raw is not None and auc_raw_plus_emb is not None:
    print(f'\nLift from adding embeddings: {auc_raw_plus_emb - auc_raw:+.4f} AUC')

## 5. Latent-space visualization

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

emb_2d = PCA(n_components=2).fit_transform(emb)
plt.figure(figsize=(6, 5))
plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=recon_error, cmap='viridis', s=8, alpha=0.6)
plt.colorbar(label='reconstruction error')
plt.title('Autoencoder latent space (PCA to 2D), colored by anomaly score')
plt.show()

## 6. Write-up (fill in)
Is the recall@5% result good enough for a human-review queue at Lumina's volume? Did the
embeddings measurably help the downstream model? What would you flag about staleness — how
would you know if the reconstruction-error threshold needs recalibrating later?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 7: Autoencoders & embeddings*